In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "accelerate",
    "soundfile",
    "huggingface_hub",
    "sentencepiece",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
subprocess.run(["apt-get", "install", "-y", "-q", "libsndfile1", "ffmpeg"], check=True)

print("✅ Environment ready")

Reading package lists...
Building dependency tree...
Reading state information...
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
✅ Environment ready


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Token")


In [4]:
HF_TOKEN        = secret_value_0

MODEL_ID        = "facebook/seamless-m4t-v2-large"
DATASET_REPO_ID = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"
OUTPUT_REPO_ID  = "hasans090/seamless_inference_lp"

DATASET_AUDIO_FOLDER = "data"
LOCAL_AUDIO_CACHE    = "/kaggle/working/audio_cache"

# ── Version control ───────────────────────────────────────────────────────────
VERSION          = 2      # ← change this before each "Save & Run All"
FILES_PER_VERSION = 205   # 102 * 10 versions = 1020, covers all 1019 files

OUTPUT_CSV = f"/kaggle/working/seamless_lipighor_v{VERSION}.csv"

TGT_LANG    = "ben"
CHUNK_SEC   = 20
OVERLAP_SEC = 2
BATCH_SIZE  = 8

import os
os.makedirs(LOCAL_AUDIO_CACHE, exist_ok=True)
print(f"✅ Config ready — Version {VERSION}, files {(VERSION-1)*FILES_PER_VERSION + 1} to {VERSION*FILES_PER_VERSION}")

✅ Config ready — Version 2, files 206 to 410


In [5]:
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
import torch

print(f"Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model on GPU 0...")
model_0 = SeamlessM4Tv2ForSpeechToText.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
model_0 = model_0.to("cuda:0")
model_0.eval()

print("Loading model on GPU 1...")
model_1 = SeamlessM4Tv2ForSpeechToText.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
model_1 = model_1.to("cuda:1")
model_1.eval()

models = [model_0, model_1]
print(f"✅ Both models loaded")
print(f"   GPU 0: {torch.cuda.get_device_name(0)}")
print(f"   GPU 1: {torch.cuda.get_device_name(1)}")

Loading processor...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

...

Loading model on GPU 1...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1429 [00:00<?, ?it/s]

SeamlessM4Tv2ForSpeechToText LOAD REPORT from: facebook/seamless-m4t-v2-large
Key                                                                           | Status     |  | 
------------------------------------------------------------------------------+------------+--+-
text_encoder.layers.{0...23}.self_attn.k_proj.weight                          | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.out_proj.weight                        | UNEXPECTED |  | 
vocoder.hifi_gan.resblocks.{0...14}.convs2.{0, 1, 2}.bias                     | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.v_proj.bias                            | UNEXPECTED |  | 
vocoder.dur_predictor.ln2.bias                                                | UNEXPECTED |  | 
t2u_model.model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.q_proj.weight     | UNEXPECTED |  | 
text_encoder.layers.{0...23}.ffn_layer_norm.weight                            | UNEXPECTED |  | 
vocoder.hifi_gan.resblocks.{0...14}.convs1.{0, 1,

✅ Both models loaded
   GPU 0: Tesla T4
   GPU 1: Tesla T4


In [6]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(
    repo_id   = OUTPUT_REPO_ID,
    repo_type = "dataset",
    private   = True,
    exist_ok  = True,
)
print(f"✅ Repo ready: https://huggingface.co/datasets/{OUTPUT_REPO_ID}")

✅ Repo ready: https://huggingface.co/datasets/hasans090/seamless_inference_lp


In [7]:
import os, torch, tempfile, subprocess, soundfile as sf
from pathlib import Path
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download, list_repo_files, upload_file

def get_duration(path):
    result = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ], capture_output=True, text=True)
    return float(result.stdout.strip())

def extract_chunk_ffmpeg(audio_path, start_sec, duration_sec, target_sr=16000):
    tmp = tempfile.mktemp(suffix=".wav")
    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start_sec),
        "-t",  str(duration_sec),
        "-i",  str(audio_path),
        "-ar", str(target_sr),
        "-ac", "1", tmp
    ], capture_output=True, check=True)
    return tmp

def dedup_overlap(t1, t2, max_w=8):
    w1, w2 = t1.split(), t2.split()
    for n in range(min(max_w, len(w1), len(w2)), 0, -1):
        if w1[-n:] == w2[:n]:
            return n
    return 0

def list_audio_files_on_hf(dataset_repo, folder, token):
    AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
    all_files  = list_repo_files(dataset_repo, repo_type="dataset", token=token)
    return sorted([
        f for f in all_files
        if f.startswith(folder + "/") and Path(f).suffix.lower() in AUDIO_EXTS
    ])

def download_single_audio(dataset_repo, hf_path, local_dir, token):
    return hf_hub_download(
        repo_id   = dataset_repo,
        filename  = hf_path,
        repo_type = "dataset",
        token     = token,
        local_dir = local_dir,
    )

def push_csv_to_hf(local_csv, output_repo, token):
    upload_file(
        path_or_fileobj = local_csv,
        path_in_repo    = Path(local_csv).name,
        repo_id         = output_repo,
        repo_type       = "dataset",
        token           = token,
    )

def transcribe_on_gpu(model, wav_files, batch_size, tgt_lang):
    all_texts = []
    for i in range(0, len(wav_files), batch_size):
        batch_files = wav_files[i : i + batch_size]

        arrays = []
        for f in batch_files:
            audio, _ = sf.read(f)
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            arrays.append(audio.astype(np.float32))

        inputs = processor(
            audio         = arrays,
            sampling_rate  = 16000,
            return_tensors = "pt",
            padding        = True,
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            output_tokens = model.generate(**inputs, tgt_lang=tgt_lang)

        texts = processor.batch_decode(output_tokens, skip_special_tokens=True)
        all_texts.extend(texts)

    return all_texts

def transcribe_dual_gpu(tmp_files, models, batch_size, tgt_lang):
    mid   = len(tmp_files) // 2
    half0 = tmp_files[:mid]
    half1 = tmp_files[mid:]

    with ThreadPoolExecutor(max_workers=2) as ex:
        f0 = ex.submit(transcribe_on_gpu, models[0], half0, batch_size, tgt_lang)
        f1 = ex.submit(transcribe_on_gpu, models[1], half1, batch_size, tgt_lang)
        res0 = f0.result()
        res1 = f1.result()

    return res0 + res1

print("✅ Utilities ready")

✅ Utilities ready


Previous

In [8]:
# hf_audio_paths = list_audio_files_on_hf(DATASET_REPO_ID, DATASET_AUDIO_FOLDER, HF_TOKEN)
# TOTAL = len(hf_audio_paths)
# print(f"Found {TOTAL} audio files\n")

# # ── Resume logic ──────────────────────────────────────────────────────────────
# def load_existing_csv_from_hf(output_repo, csv_filename, token):
#     try:
#         path = hf_hub_download(
#             repo_id        = output_repo,
#             filename       = csv_filename,
#             repo_type      = "dataset",
#             token          = token,
#             force_download = True,
#         )
#         df = pd.read_csv(path)
#         print(f"✅ Loaded from HF — {len(df)} rows done so far")
#         return df
#     except Exception as e:
#         print(f"ℹ️ No existing CSV on HF (starting fresh): {e}")
#         return None

# hf_df = load_existing_csv_from_hf(OUTPUT_REPO_ID, Path(OUTPUT_CSV).name, HF_TOKEN)

# if hf_df is not None:
#     done_ids = set(hf_df["id"].astype(str).tolist())
#     results  = hf_df.to_dict("records")
#     hf_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
# elif Path(OUTPUT_CSV).exists():
#     done_df  = pd.read_csv(OUTPUT_CSV)
#     done_ids = set(done_df["id"].astype(str).tolist())
#     results  = done_df.to_dict("records")
#     print(f"Resuming from local — {len(done_ids)} done")
# else:
#     done_ids = set()
#     results  = []
#     print("Starting fresh")

# print(f"\n📊 {len(done_ids)} done / {TOTAL} total — {TOTAL - len(done_ids)} remaining\n")

# # ── Loop ──────────────────────────────────────────────────────────────────────
# for i, hf_path in enumerate(hf_audio_paths):
#     file_id = Path(hf_path).stem

#     if file_id in done_ids:
#         continue

#     os.system("clear")
#     print(f"{'─'*60}")
#     print(f"  [{len(results)+1}/{TOTAL}]  {file_id}")
#     print(f"{'─'*60}\n")

#     try:
#         print("⬇  Downloading...")
#         audio_path = download_single_audio(DATASET_REPO_ID, hf_path, LOCAL_AUDIO_CACHE, HF_TOKEN)

#         total_sec = get_duration(audio_path)
#         print(f"⏱  Duration: {total_sec:.0f}s ({total_sec/60:.1f} min)")

#         step = CHUNK_SEC - OVERLAP_SEC
#         starts, pos = [], 0.0
#         while pos < total_sec:
#             starts.append(pos)
#             if pos + CHUNK_SEC >= total_sec: break
#             pos += step
#         print(f"🔪  Chunks: {len(starts)}  →  {len(starts)//2} | {len(starts) - len(starts)//2} across 2 GPUs\n")

#         tmp_files = []
#         for start in starts:
#             dur = min(CHUNK_SEC, total_sec - start)
#             tmp_files.append(extract_chunk_ffmpeg(audio_path, start, dur))

#         chunk_texts = transcribe_dual_gpu(tmp_files, models, BATCH_SIZE, TGT_LANG)

#         for tmp in tmp_files:
#             os.remove(tmp)

#         for j, text in enumerate(chunk_texts):
#             print(f"  chunk {j+1:>3}/{len(starts)}: {text[:70]}")

#         words = chunk_texts[0].split() if chunk_texts else []
#         for k in range(1, len(chunk_texts)):
#             skip = dedup_overlap(chunk_texts[k-1], chunk_texts[k]) if OVERLAP_SEC > 0 else 0
#             words.extend(chunk_texts[k].split()[skip:])
#         transcript = " ".join(words).strip()

#         os.remove(audio_path)
#         print(f"\n✅  {len(transcript.split())} words total")

#     except Exception as ex:
#         transcript = ""
#         print(f"\n❌  ERROR: {ex}")

#     results.append({"id": file_id, "transcript": transcript})
#     pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

#     files_done = len(results)
#     is_last    = (i == TOTAL - 1)
#     if files_done % 10 == 0 or is_last:
#         try:
#             push_csv_to_hf(OUTPUT_CSV, OUTPUT_REPO_ID, HF_TOKEN)
#             print(f"📤  CSV pushed to HF ({files_done} rows)")
#         except Exception as e:
#             print(f"⚠️  Push failed: {e}")

# print(f"\n✅ All done — {len(results)} rows")
# print(pd.read_csv(OUTPUT_CSV)[["id", "transcript"]].head(10).to_string(index=False))

In [9]:
#New
hf_audio_paths = list_audio_files_on_hf(DATASET_REPO_ID, DATASET_AUDIO_FOLDER, HF_TOKEN)
TOTAL_ALL = len(hf_audio_paths)

# ── Slice for this version ────────────────────────────────────────────────────
start_idx = (VERSION - 1) * FILES_PER_VERSION
end_idx   = min(VERSION * FILES_PER_VERSION, TOTAL_ALL)
my_files  = hf_audio_paths[start_idx:end_idx]
TOTAL     = len(my_files)

print(f"Total files in repo : {TOTAL_ALL}")
print(f"This version (v{VERSION}): files [{start_idx}:{end_idx}] — {TOTAL} files\n")

# ── Resume logic ──────────────────────────────────────────────────────────────
def load_existing_csv_from_hf(output_repo, csv_filename, token):
    try:
        path = hf_hub_download(
            repo_id        = output_repo,
            filename       = csv_filename,
            repo_type      = "dataset",
            token          = token,
            force_download = True,
        )
        df = pd.read_csv(path)
        print(f"✅ Loaded from HF — {len(df)} rows done so far")
        return df
    except Exception as e:
        print(f"ℹ️ No existing CSV on HF (starting fresh): {e}")
        return None

hf_df = load_existing_csv_from_hf(OUTPUT_REPO_ID, Path(OUTPUT_CSV).name, HF_TOKEN)

if hf_df is not None:
    done_ids = set(hf_df["id"].astype(str).tolist())
    results  = hf_df.to_dict("records")
    hf_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
elif Path(OUTPUT_CSV).exists():
    done_df  = pd.read_csv(OUTPUT_CSV)
    done_ids = set(done_df["id"].astype(str).tolist())
    results  = done_df.to_dict("records")
    print(f"Resuming from local — {len(done_ids)} done")
else:
    done_ids = set()
    results  = []
    print("Starting fresh")

print(f"\n📊 {len(done_ids)} done / {TOTAL} total — {TOTAL - len(done_ids)} remaining\n")

# ── Loop ──────────────────────────────────────────────────────────────────────
for i, hf_path in enumerate(my_files):
    file_id = Path(hf_path).stem

    if file_id in done_ids:
        continue

    os.system("clear")
    print(f"{'─'*60}")
    print(f"  [v{VERSION} — {len(results)+1}/{TOTAL}]  {file_id}")
    print(f"{'─'*60}\n")

    try:
        print("⬇  Downloading...")
        audio_path = download_single_audio(DATASET_REPO_ID, hf_path, LOCAL_AUDIO_CACHE, HF_TOKEN)

        total_sec = get_duration(audio_path)
        print(f"⏱  Duration: {total_sec:.0f}s ({total_sec/60:.1f} min)")

        step = CHUNK_SEC - OVERLAP_SEC
        starts, pos = [], 0.0
        while pos < total_sec:
            starts.append(pos)
            if pos + CHUNK_SEC >= total_sec: break
            pos += step
        print(f"🔪  Chunks: {len(starts)}  →  {len(starts)//2} | {len(starts) - len(starts)//2} across 2 GPUs\n")

        tmp_files = []
        for start in starts:
            dur = min(CHUNK_SEC, total_sec - start)
            tmp_files.append(extract_chunk_ffmpeg(audio_path, start, dur))

        chunk_texts = transcribe_dual_gpu(tmp_files, models, BATCH_SIZE, TGT_LANG)

        for tmp in tmp_files:
            os.remove(tmp)

        for j, text in enumerate(chunk_texts):
            print(f"  chunk {j+1:>3}/{len(starts)}: {text[:70]}")

        words = chunk_texts[0].split() if chunk_texts else []
        for k in range(1, len(chunk_texts)):
            skip = dedup_overlap(chunk_texts[k-1], chunk_texts[k]) if OVERLAP_SEC > 0 else 0
            words.extend(chunk_texts[k].split()[skip:])
        transcript = " ".join(words).strip()

        os.remove(audio_path)
        print(f"\n✅  {len(transcript.split())} words total")

    except Exception as ex:
        transcript = ""
        print(f"\n❌  ERROR: {ex}")

    results.append({"id": file_id, "transcript": transcript})
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    files_done = len(results)
    is_last    = (i == TOTAL - 1)
    if files_done % 10 == 0 or is_last:
        try:
            push_csv_to_hf(OUTPUT_CSV, OUTPUT_REPO_ID, HF_TOKEN)
            print(f"📤  CSV pushed to HF ({files_done} rows) — seamless_lipighor_v{VERSION}.csv")
        except Exception as e:
            print(f"⚠️  Push failed: {e}")

print(f"\n✅ Version {VERSION} done — {len(results)} rows")

Total files in repo : 1019
This version (v2): files [205:410] — 205 files

ℹ️ No existing CSV on HF (starting fresh): 404 Client Error. (Request ID: Root=1-69d0c380-2d4ab8525dbd5a6e0f76ce9e;6922e429-28a9-46be-8ca2-1dc0b29f065d)

Entry Not Found for url: https://huggingface.co/datasets/hasans090/seamless_inference_lp/resolve/main/seamless_lipighor_v2.csv.
Starting fresh

📊 0 done / 205 total — 205 remaining

────────────────────────────────────────────────────────────
  [v2 — 1/205]  BHLasI0xL1o
────────────────────────────────────────────────────────────

⬇  Downloading...


data/BHLasI0xL1o.mp3:   0%|          | 0.00/43.6M [00:00<?, ?B/s]

⏱  Duration: 3158s (52.6 min)
🔪  Chunks: 176  →  88 | 88 across 2 GPUs

  chunk   1/176: বন্ধুরা নমস্কার, এসো গল্প শুনে ইউটিউব চ্যানেলে আমি কামাল আপনাদের সবাইক
  chunk   2/176: বন্ধুরা যারা আজ আমার চ্যানেলে নতুন এসেছেন তাদের উদ্দেশ্য বলছি আপনি যদি
  chunk   3/176: টিপ দিতে ভুলবেন না যাতে আমার চ্যানেলে যে কোন নোটিফিকেশন আপলোড করা হয় 
  chunk   4/176: আমাকে সমৃদ্ধ করবেন এবং পারলে বন্ধুদের শেয়ার করবেন ভিডিওটি এই কটি কথা 
  chunk   5/176: ১৩। ইউনিভার্সিটিতে ঢোকার মুখেই বিমানের সঙ্গে দেখা
  chunk   6/176: মুখের সাথে বিমানের সাথে দেখা করে তাকে দেখে হাত নামিয়ে ডাকল বিমান বলল 
  chunk   7/176: পাঠিয়েছি তারা এসে বলেছে তুমি নেই গতকাল তোমার হোস্টেলে গিয়ে পাওয়া যা
  chunk   8/176: কতগুলো প্রশ্ন করতে পারে এবং তার উত্তরের উত্তর পাওয়া যায় দ্বিতীয় শ্র
  chunk   9/176: ছদ্মবেশী বই থেকে ছড়িয়ে পড়লে সেই বিষয়ের থিসিস হয়ে যায় আর যার অধীন
  chunk  10/176: পরিচয় দেয় না বিশেষ করে বাংলায় অ্যানিমেশ বিমানকে বলল চলো ছেলেমেয়েরা
  chunk  11/176: মাঝে মাঝে কিছুদিন ধরেই অস্থিরতা দেখা যাচ্ছে যখন সে ত

data/BIxL_VThMNk.mp3:   0%|          | 0.00/74.6M [00:00<?, ?B/s]

⏱  Duration: 4899s (81.7 min)
🔪  Chunks: 273  →  136 | 137 across 2 GPUs

  chunk   1/273: কেউ দেখিনি তো আলাইয়াহ আছে মানিমাল আলাইয়াহ আমি কিন্তু সহজে হাঁটতে পার
  chunk   2/273: পাঁও বছর পাঁও আমি রোষে রোশে রোসে খাওয়ার খেতে খেতে খাওয়ার খেতে খেতে খ
  chunk   3/273: বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ বন্ধ 
  chunk   4/273: এই মুরসা
  chunk   5/273: হ্যাঁ মায়া আমার এই জীবনে আমার কোন পুত্র নেই তুমি এই যে তুই তুই তুই তো
  chunk   6/273: এই যে, এই যে, এই যে, এই যে, এই যে, এই যে, এই যে, এই যে, এই যে, এই যে, 
  chunk   7/273: এইটা কি আমার মেয়ে বাবার সাথে কথা বলবো কি
  chunk   8/273: এই সিরিয়ালের রোমান্টিক দৃশ্য এইটা কি আর দেখো না
  chunk   9/273: আমি বলবো আমি আধুনিকা বলটা ধরতে চাইলে তুই এদিকে বসো না
  chunk  10/273: এইবার কও আমার গলু কলি কি আর না না
  chunk  11/273: আরে আমি তোমার দুল ভাই আরে তুই তো কুস্তি ভাই আরে তুমি তো মাও বলতো যে কথ
  chunk  12/273: এই বুঝেছো এইটা কি করে করবি আরে কইরো একটা পাতা একটা চাষ আর একটা খাড়া হ
  chunk  13/273: দেখো না তোমার সীমানা

...

data/ODAv4krbs4Y.mp3:   0%|          | 0.00/36.5M [00:00<?, ?B/s]

⏱  Duration: 2462s (41.0 min)
🔪  Chunks: 137  →  68 | 69 across 2 GPUs

  chunk   1/137: Where's your face? Where's your face? Where's your face?
  chunk   2/137: ছিল বাঘের মাথায় পড়েছিল ওরা ভয় পেয়ে পালিয়েছিল শত্রু সেনা ও ওহ এখান
  chunk   3/137: ♪ Ooh, ooh, ooh, ooh, ooh, ooh, ooh, ooh, ooh, ooh, ooh, ooh, ooh, ooh
  chunk   4/137: "আচ্ছা, আমি কি জানি, আমি কি জানি, আমি কি জানি, আমি কি জানি, আমি কি জান
  chunk   5/137: [সঙ্গীতের আওয়াজ]
  chunk   6/137: এই যে এই যে এই যে এই যে এই যে এই যে এই যে এই যে এই যে এই যে এই যে এই য
  chunk   7/137: আহ তোমার কি হবে ম কেন রক্ত ভেজায়ে না মাতি
  chunk   8/137: এই রকমাটি
  chunk   9/137: এইমাত্র পাওয়া বাংলা খবর। Bangla News 23 Jan 2022 | Bangladesh Latest 
  chunk  10/137: আজকে তো তোমার মুখে নাচায় রানা আকাংক্ষার তালু ঝাড়ু
  chunk  11/137: "নাগাদ ফাইটিং সিনেমারেও"
  chunk  12/137: এইমাত্র পাওয়া বাংলা খবর। Bangla News 23rd September 2022 |Bangladesh 
  chunk  13/137: আমারে দেখিস না
  chunk  14/137: ♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪♪

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

📤  CSV pushed to HF (205 rows) — seamless_lipighor_v2.csv

✅ Version 2 done — 205 rows
